# Chatterbox v3: three ways to render a group of segments

The pipeline joins several sentences into one prompt and calls `generate` once. That is
where the audio cut-offs come from: upstream hardcodes `max_new_tokens=1000`, and at 25
speech tokens per second that is a 40 second ceiling, after which the last sentence stops
mid word. This notebook renders the same text three ways so the options can be compared by
ear and by numbers before the app commits to one.

| Section | What happens |
|---|---|
| 0. Environment setup | conda and uv commands to create the kernel, run once |
| 1. Model location and preparation | pick CPU or GPU, pick the checkpoint, see where the weights live, load them |
| 2. Data preparation | reference voice, sample text, segmentation into batches |
| 3. Inference | sampling settings, then each variant rendered to wav |
| 4. Result metrics | per segment and per variant tables, saved to json |
| 5. Listen and analysis | audio players and what to judge |

**Standalone.** Nothing is imported from the ebook2audiobook repo; every helper is defined
in the notebook, and only the packages in section 0 are needed.

## 0. Environment setup

Run once in a terminal on the machine that will run the notebook, then pick the
**Python (chat11, chatterbox v3)** kernel. conda provides only Python and uv; uv installs the
packages.

```bash
conda create -n chat11 python=3.11 -y
conda activate chat11
conda install -c conda-forge uv -y
uv pip install --python "$CONDA_PREFIX/bin/python" \
    "chatterbox-tts @ git+https://github.com/resemble-ai/chatterbox.git@65b18437" \
    torchaudio soundfile numpy psutil tiktoken huggingface_hub ipykernel
python -m ipykernel install --user --name chat11 --display-name "Python (chat11, chatterbox v3)"
python -c "import torch, torch._dynamo, chatterbox.mtl_tts; print(torch.__version__, torch.cuda.is_available())"
```

| Resolved to | Why |
|---|---|
| chatterbox-tts at git `65b18437` | the PyPI release with the same version number (0.1.7) has no v3 loader |
| torch / torchaudio 2.6.0, CUDA 12.4 wheels | chatterbox's own pin; needs an NVIDIA driver of 550 or newer for GPU, works on CPU without one |
| transformers 5.2.0, diffusers 0.29.0, numpy 1.26 | chatterbox's own pins |

The last command should print `2.6.0+cu124 True` on a GPU machine (`False` on CPU only). If
`torch._dynamo` fails with "partially initialized module", restart the kernel first: that
message usually hides an earlier import error in the same session.

Inside the ebook2audiobook repo the same tree comes from `uv sync --extra chatterbox --extra notebook`.

In [ ]:
import json
import os
import re
import resource
import time
from dataclasses import dataclass, field, replace
from pathlib import Path

import numpy as np
import psutil
import soundfile as sf
import tiktoken
import torch
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from IPython.display import Audio, display

## 1. Model location and preparation

Chatterbox is one Hugging Face repo, `ResembleAI/chatterbox`, holding two checkpoints:

| Checkpoint | Languages | Files |
|---|---|---|
| `english` | en only | `ve.safetensors`, `t3_cfg.safetensors`, `s3gen.safetensors`, `tokenizer.json` |
| `multilingual` | 23 | `ve.pt`, `s3gen.pt`, the grapheme tokenizer, plus **one T3 shard per version**: `t3_mtl23ls_v2.safetensors` or `t3_mtl23ls_v3.safetensors` |

v2 and v3 differ only in that T3 shard, so both versions share one directory and switching
costs one extra download. Weights go to the `model_dir` you set in 1.2, as plain files, never the Hugging Face cache.

### 1.1 Hardware on this machine

In [ ]:
print(f"cpu cores        {os.cpu_count()}")
print(f"torch threads    {torch.get_num_threads()}")
print(f"cuda available   {torch.cuda.is_available()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}           {props.name}, {props.total_memory / 1024 ** 3:.1f} GB")

### 1.2 Settings: checkpoint location, device and limits

This is the only cell to edit in this section.

| Field | Values | Meaning |
|---|---|---|
| `model_dir` | path, **required** | folder holding the checkpoint. There is no default: set it to wherever the files are, or wherever they should go. If the folder already has them they load as is, with no download; only missing files are fetched, into this folder. |
| `model_type` | `"english"`, `"multilingual"` | which checkpoint. v3 exists only for `multilingual`. |
| `mtl_version` | `"v2"`, `"v3"` | multilingual only, which T3 shard. |
| `device` | `"cpu"`, `"cuda"`, `"cuda:1"`, `"auto"` | `"cpu"` runs everything on the CPU, even when a GPU is present. `"auto"` takes `cuda` when available, else CPU. |
| `cpu_threads` | count | CPU only. Cores torch may use. 4 is the cap upstream ebook2audiobook applies in CPU mode (`OMP_NUM_THREADS=4`). Raise it toward the core count for speed, lower it to leave cores free. |
| `max_memory_gb` | GB | Memory budget; 11 is the app's default `--max_vram`. On CUDA it is a hard ceiling. On CPU torch has no RAM ceiling, so, as upstream does, it is checked against free system RAM and you get a warning before loading. |

For example, CPU only: `ModelSetup(model_dir="/data/models/chatterbox", device="cpu", cpu_threads=8, max_memory_gb=8)`.
On CPU, expect every render below to take several times longer than the audio it produces.

In [ ]:
@dataclass
class ModelSetup:
    model_dir: str
    model_type: str = "multilingual"
    mtl_version: str = "v3"
    device: str = "auto"
    cpu_threads: int = 4
    max_memory_gb: int = 11


model_setup = ModelSetup(model_dir="")
model_setup

### 1.3 Resolve the device and apply its limits

Same mechanism as the app. `memory_report_gb` reads free and total memory: GPU memory from
`torch.cuda.mem_get_info` on CUDA, system RAM from `psutil` on CPU. On CUDA the budget becomes
`torch.cuda.set_per_process_memory_fraction`, a hard ceiling. On CPU `torch.set_num_threads`
caps the cores and the budget is only checked against free RAM. Both must happen before any
weights are loaded. `resolve_device` stops early with a clear message when CUDA is requested
but torch cannot see a GPU.

In [ ]:
def resolve_device(requested: str) -> str:
    if requested == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cpu":
        return "cpu"
    if not torch.cuda.is_available():
        raise RuntimeError(
            f"device='{requested}' but torch sees no CUDA device. "
            "Use device='cpu', or install a CUDA build of torch."
        )
    return requested


def memory_report_gb(device: str) -> tuple[str, float, float]:
    if device.startswith("cuda"):
        index = torch.device(device).index or 0
        free, total = torch.cuda.mem_get_info(index)
        return torch.cuda.get_device_name(index), free / 1024 ** 3, total / 1024 ** 3
    ram = psutil.virtual_memory()
    return "System RAM", ram.available / 1024 ** 3, ram.total / 1024 ** 3


def apply_device_limits(device: str, setup: ModelSetup) -> None:
    name, free_gb, total_gb = memory_report_gb(device)
    print(f"{name}: {free_gb:.2f} GB free of {total_gb:.2f} GB")
    if free_gb < setup.max_memory_gb:
        print(f"warning: less free memory than max_memory_gb={setup.max_memory_gb}, loading may fail or swap")
    if device == "cpu":
        torch.set_num_threads(setup.cpu_threads)
        print(f"cpu: torch limited to {torch.get_num_threads()} of {os.cpu_count()} cores")
        return
    if total_gb <= setup.max_memory_gb:
        print(f"{device}: card is {total_gb:.2f} GB, already under the {setup.max_memory_gb} GB budget")
        return
    torch.cuda.set_per_process_memory_fraction(setup.max_memory_gb / total_gb, torch.device(device).index or 0)
    print(f"{device}: capped to {setup.max_memory_gb} GB of {total_gb:.2f} GB")


device = resolve_device(model_setup.device)
apply_device_limits(device, model_setup)

### 1.4 Weights on disk

`checkpoint_files` names every file a checkpoint needs (required) or reads when present
(optional). The list below shows which of them `model_dir` already has.

In [ ]:
def checkpoint_files(model_type: str, mtl_version: str) -> tuple[list[str], list[str]]:
    if model_type == "english":
        return ["ve.safetensors", "t3_cfg.safetensors", "s3gen.safetensors", "tokenizer.json"], ["conds.pt"]
    if model_type != "multilingual":
        raise ValueError(f"model_type must be 'english' or 'multilingual', got '{model_type}'")
    required = ["ve.pt", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json", f"t3_mtl23ls_{mtl_version}.safetensors"]
    return required, ["conds.pt", "Cangjie5_TC.json"]


def describe_weights(model_dir: str, model_type: str, mtl_version: str) -> None:
    if not model_dir:
        raise ValueError("model_dir is not set: put the checkpoint folder in ModelSetup in 1.2")
    required, optional = checkpoint_files(model_type, mtl_version)
    print(f"model_dir {model_dir}")
    for name in required + optional:
        file_path = Path(model_dir) / name
        kind = "required" if name in required else "optional"
        size = f"{file_path.stat().st_size / 1024 ** 2:9.1f} MB" if file_path.exists() else "  missing   "
        print(f"  {size}  {kind:8s}  {name}")


describe_weights(model_setup.model_dir, model_setup.model_type, model_setup.mtl_version)

### 1.5 Load the model

`load_model` takes the checkpoint folder explicitly. When every required file is already
there, it loads straight from that folder and never touches the network. Otherwise it first
downloads only the missing files from `ResembleAI/chatterbox` into that same folder,
as real files, so deleting the model is deleting the folder.

Memory is measured right after loading, so later cells can report what inference costs on top
of the weights. On CUDA that is peak allocated VRAM. On CPU it is the process resident high
water mark, which only ever grows.

In [ ]:
def reset_peak_memory(device: str) -> None:
    if device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats(device)


def peak_memory_gb(device: str) -> float:
    if device.startswith("cuda"):
        return torch.cuda.max_memory_allocated(device) / 1024 ** 3
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 ** 2


def fetch_missing(model_dir: str, model_type: str, mtl_version: str) -> None:
    required, optional = checkpoint_files(model_type, mtl_version)
    missing = [name for name in required if not (Path(model_dir) / name).exists()]
    if not missing:
        return
    print(f"downloading {', '.join(missing)} into {model_dir}")
    snapshot_download("ResembleAI/chatterbox", local_dir=model_dir, allow_patterns=required + optional)


def load_model(model_dir: str, model_type: str, mtl_version: str, device: str) -> object:
    fetch_missing(model_dir, model_type, mtl_version)
    print(f"loading {model_type} {mtl_version if model_type != 'english' else ''} from {model_dir} on {device}")
    if model_type == "english":
        from chatterbox.tts import ChatterboxTTS
        return ChatterboxTTS.from_local(ckpt_dir=model_dir, device=device)
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    return ChatterboxMultilingualTTS.from_local(ckpt_dir=model_dir, device=device, t3_model=mtl_version)


reset_peak_memory(device)
model = load_model(model_setup.model_dir, model_setup.model_type, model_setup.mtl_version, device)
weights_gb = peak_memory_gb(device)
print(f"{type(model).__name__}, sample rate {model.sr} Hz, memory after load {weights_gb:.2f} GB")

## 2. Data preparation

### 2.1 Settings: voice, language and batching

| Field | Meaning |
|---|---|
| `reference_audio` | path to a 5 to 20 second wav to clone, or `None` for the builtin voice in `conds.pt` |
| `language_id` | language of the sample text, one of the 23 codes; ignored by `english` |
| `max_tokens_per_segment` | segmenter budget per segment, the pipeline default |
| `segments_per_batch` | how many segments form one group, the unit every variant renders |

In [ ]:
@dataclass
class DataSetup:
    reference_audio: str | None = None
    language_id: str = "en"
    max_tokens_per_segment: int = 48
    segments_per_batch: int = 8


data_setup = DataSetup()
data_setup

### 2.2 Reference voice

Conditioning is computed once from the reference clip and reused by every render below.

In [ ]:
if data_setup.reference_audio:
    model.prepare_conditionals(data_setup.reference_audio)
    print(f"voice cloned from {data_setup.reference_audio}")
else:
    print("builtin voice from conds.pt")

### 2.3 Sample text

Both corpora live in one `text_sample`: the money and ice-age passages first, then the
astrophysics passages. `clean_corpus` straightens curly quotes, drops footnote numbers after a
full stop, and unwraps single line breaks while keeping paragraph breaks. The cleaned text is
what gets segmented and rendered.

In [ ]:
import re

def clean_corpus(text: str) -> str:
    # Normalize curly/smart quotes.
    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )
    # Remove digits immediately following a period: "word.123" -> "word."
    text = re.sub(r"\.(\d+)", ".", text)
    # Preserve paragraph breaks (2+ newlines), but unwrap single line breaks.
    parts = re.split(r"(\n{2,})", text)
    for index in range(0, len(parts), 2):
        parts[index] = re.sub(r"[ \t]*\n[ \t]*", " ", parts[index])
    return "".join(parts)


text_sample = """
The challenge for us is that no amount of studying or open-mindedness can
genuinely recreate the power of fear and uncertainty
I can read about what it was like to lose everything during the Great Depression. But
I don’t have the emotional scars of those who actually experienced it. And the person
who lived through it can’t fathom why someone like me could come across as
complacent about things like owning stocks. We see the world through a different
lens.
Spreadsheets can model the historic frequency of big stock market declines. But
they can’t model the feeling of coming home, looking at your kids, and wondering if
you’ve made a mistake that will impact their lives. Studying history makes you feel
like you understand something. But until you’ve lived through it and personally felt
its consequences, you may not understand it enough to change your behavior.
We all think we know how the world works. But we’ve all only experienced a tiny
sliver of it.


Dogs were domesticated 10,000 years ago and still retain some behaviors of their
wild ancestors. Yet here we are, with between 20 and 50 years of experience in the
modern financial system, hoping to be perfectly acclimated.
For a topic that is so influenced by emotion versus fact, this is a problem. And it
helps explain why we don’t always do what we’re supposed to with money.
We all do crazy stuff with money, because we’re all relatively new to this game and
what looks crazy to you might make sense to me. But no one is crazy—we all make
decisions based on our own unique experiences that seem to make sense to us in a
given moment.

Modern capitalism is a pro at two things: generating wealth and generating envy.
Perhaps they go hand in hand; wanting to surpass your peers can be the fuel of hard
work. But life isn’t any fun without a sense of enough. Happiness, as it’s said, is just
results minus expectations.

The only way to know how much food you can eat is to eat until you’re sick. Few try
this because vomiting hurts more than any meal is good. For some reason the same
logic doesn’t translate to business and investing, and many will only stop reaching
for more when they break and are forced to. This can be as innocent as burning out
at work or a risky investment allocation you can’t maintain. On the other end there’s
Rajat Guptas and Bernie Madoffs in the world, who resort to stealing because every
dollar is worth reaching for regardless of consequence.

There were plenty of theories about why ice ages occurred. To account for their
enormous geological influence the theories were equally grand. The uplifting of
mountain ranges, it was thought, may have shifted the Earth’s winds enough to alter
the climate. Others favored the idea that ice was the natural state, interrupted by
massive volcanic eruptions that warmed us up. But none of these theories could account for the cycle of ice ages. The growth of
mountain ranges or some massive volcano may explain one ice age. It could not
explain the cyclical repetition of five.

The astrophysicist, however, does not make the missiles or the bombs.
Astrophysicists make no weapons at all. Instead, we and the military happen to care
about many of the same things: multi-spectral detection, ranging, tracking, imaging,
high ground, nuclear fusion, access to space. 
Looking down at Earth from an orbiting spacecraft, a rational person could certainly
feel that "necessary defence" may have more to do with the vulnerability of our
beautiful blue planet, exposed to all the vicissitudes of the cosmos, than with the
transient power of a single country’s weapons, policymakers, nationalists, and
ideologues, however virulent. From hundreds of kilometers above the surface of the
globe, "Peace on Earth, Goodwill Toward Men" might sound less like a standard line
on a Christmas card and more like an essential step toward a viable future, in which
all of humankind cooperates in protecting Earth from the enemies among us and the
threats above us.
But none of those expressions of national pride and militarism meshed with my
sense of armed conflict. I didn’t understand how they fit together. That
twenty-something grease monkey did, though. He was plugged into a primal passion
that has energized so many wars across the millennia. Just not the war I’d grown up
with.
With the winding-down of the Cold War, the aerospace industry had undergone
unrelenting shrinkage and consolidation. Seventy-five aerospace companies were in
operation on the day Reagan was elected, merging into sixty-one by the time the
Berlin Wall fell, and finally into just five titans—Lockheed Martin, Boeing, Raytheon,
Northrop Grumman, and General Dynamics—by the time the Twin Towers crumbled
into toxic dust. Some 600,000 scientific and technical jobs had vanished in just a
dozen of those years, along with incalculable quantities of experience and
intellectual capital.
Gone are the days when a nation’s security could be measured by bombers and
battleships. Security in this new era requires harnessing all of America’s strengths:
the force of our diplomacy; the might of our military; the vigor and competitiveness
of our economy; and the power of our ideals.
Has the United States funded and engaged in this kind of research purely for the
sake of discovery? Hardly. Most American accelerators were built during the Cold
War, when the particle physicist was a vital resource for increasing the lethality of
nuclear weapons. That’s how astrophysics—specifically astro-particle physics, a
branch of cosmology—became an auxiliary beneficiary of Cold War science priorities.
Astrophysics and the military are conjoined whether their shared buoy bobs up or
down in the tidewaters of politics.
Consider units of time. If the Sun never set and the Moon never waned, our measures
of time might be grounded solely in biology—the beating heart, circadian rhythms,
menstruation—because “periodicity is part of who we are.”1 But the Sun does set,
and the Moon predictably waxes and wanes. Transitions recur endlessly in the skies
above. Celestial cycles offer themselves as a natural measure of time in units we
care about.
The modern distinction between astronomy and astrology was, at one time, fuzzy
and irrelevant, just like the distinction between alchemy and chemistry or between
magic and medicine. Promoting the propitious, avoiding calamity, and forecasting
death suggest astrological interpretation. Yet prediction, which is an offshoot of
analysis, can be exacting and scientific, given the right practitioner. Accurate
observation of the skies, combined with a grasp of physics and a cartography of the
cosmos, is the cornerstone of both.
The full electromagnetic spectrum may extend forever in both directions: toward
ever longer wavelengths, perhaps limited by the size of the universe itself, and
toward ever smaller wavelengths, perhaps limited by quantum physics. Currently we
have the technology to detect wavelengths ranging from less than a
hundred-billionth of a meter (high-frequency gamma rays) to many hundreds of
kilometers (extremely low-frequency radio waves). That’s a factor of quadrillions.
Millennia ago, if you wanted to look up at the sky or across a broad valley, you might
have used a long sighting tube to focus your attention and reduce glare, as did
Aristotle and probably his predecessors. But no empty tube, however long—and
whether cast in gold by an ancient Assyrian metalworker, carved from jade by an
ancient Chinese artisan, or fastened to an armillary sphere by a mathematically
astute medieval pope1—would improve your physiological capacity to detect the
planet Neptune or assess the size of a rival army or navy massing on a distant shore.
Put a couple of lenses in the tube, though, and you’ve got yourself an optical
telescope.
A tool for augmenting the senses, the telescope enables you both to detect things
too faint to see and to resolve detail where your eyes would otherwise fail you. First,
it shows you that an object exists; then, by revealing the object’s shape, motion, and
color, it hints at what the object might be. The telescope’s task is to collect at a
distance as much visual information as it can, and feed it to your brain via your eyes.

Fortunately for astrophysicists, the germ of a better solution was already available.
In 1835 the German chemist Justus von Liebig had introduced the silvered-glass
mirror. Made by depositing a thin layer of silver vapor on one side of a slab of
polished glass, it offered an excellent image and soon became a fixture of every
bourgeois household. Two decades later, Jean-Bernard-Léon Foucault (the pendulum
fellow), in collaboration with the Paris Observatory’s official optician, improved upon
this technique by adding a subsequent phase: localized repolishing to correct errors
of form. This enabled Foucault to make ever-larger reflecting telescopes, culminating
in an eighty-centimeter telescope installed in the Marseille Observatory in 1864.71
Today the largest telescopes in the world are all reflectors, and all of them use a
mirror with a vapor-deposited metal coating on one polished glass surface. While the
lens of the largest extant refracting telescope is one meter across, the mirror of the
largest reflecting telescope is more than ten meters across. Others in the works
approach forty meters in diameter. Hardly anything limits the size of the mirror,
because it is mounted from the back. As a result, since the end of the nineteenth
century, the reflector has been the astrophysicist’s instrument of choice.


The military solution, however, lay elsewhere. For nearly the entire nineteenth
century, military planners and artillerymen alike fretted far less than astronomers
about the limited availability of fine optical glass. An infantry rifle that could be fired
effectively at a target more than a mile away was not yet on the market.72 Gunners
did not rely on barrel-mounted spotting scopes. Civil War cannons were fired point
blank in the general direction of a nearby visible enemy; battling Northerners and
Southerners estimated distances strictly by eye and aimed their guns with the aid of
spirit levels and plumb lines, hoping to overwhelm the enemy with a barrage of shot.
“The gunners sighted their fieldpieces hastily and banged away, trusting to hit some
vital spot,” writes Lieutenant Colonel F. E. Wright in a historical overview produced in
1921 for the Ordnance Department of the US Army.
The modern, Western offspring of astronomy, astrophysics is not even a century and
a half old. Its midwives were two nineteenth-century technological innovations. The
more widely known of the two, photography—literally “light drawing”—stemmed
from a welter of investigations into the image-forming proclivities of light. The
lesser-known and more specialized innovation, spectroscopy—which separates light
into its component colors, yielding heaps of information about its source—derived
from the prismatic study of the Sun’s spectrum and the discovery that every
substance radiates a characteristic and unique combination of colors. Jointly,
photography and spectroscopy empowered the astronomer to record and analyze
whatever light the available telescopes could gather from the sky.
Creating confusion seemed a better solution than trying to attain invisibility.
Suddenly artists became facilitators of military goals, as warriors picked up some of
the disintegrationist, scientistic visual strategies of the vanguard movements of
Cubism, Futurism, and Vorticism. Picasso and Braque, the fathers of Cubism, were
delighted to see what they regarded as their aesthetic invention being applied to
ships and weaponry; walking down a boulevard in Paris one evening and seeing a
convoy of zigzag-painted heavy guns heading for the front, Picasso reportedly
exclaimed, “We invented that!” Franklin D. Roosevelt himself, assistant secretary of
the US Navy during World War I, is said to have shouted, after being shown a
bedazzled test ship, “How the hell do you expect me to estimate the course of a
God-damn thing all painted up like that?” In the end, though, standardized dazzle
camouflage apparently did not live up to its promise. Attacks proceeded at similar
rates on ships with and without such a coating. Nevertheless, despite plenty of
evidence to the contrary, magical thinking about the efficacy of disruptive
camouflage persisted through World War II and beyond.
Several options for disappearing from the visible part of the spectrum have a long
wartime pedigree. The simplest is to exploit the darkness of night. Another is to blind
the enemy. Set a huge bonfire, and enemy forces who look at it will be unable to see
anything other than flames and therefore be unable to target you with any precision.
In recent decades, both lasers and smokescreens have been used to blind the
enemy: throw a white-phosphorus grenade at your target, and you’ll get an instant
smokescreen that will also scorch whoever is in the vicinity while it masks your own
maneuvers and your own infrared radiation. Blinding also happens to us in the
universe, when the light of a host star swamps the much fainter reflected light of its
exoplanets. This was a big problem until a couple of decades ago, when space
scientists began using a special occulting disk in their telescope’s optics to block out
the offending starlight, thus achieving the opposite of what a telescope was invented
to accomplish.
Another, very different approach to disappearance is transparency—embodied in the
clear glass window. Flies, moths, birds, and visitors from outer space who don’t know
about windows must be baffled by the interposition of something visually
imperceptible but impenetrable between themselves and the scenery.



Objective truths of science are not founded in belief systems. They are not established by the authority of leaders or the power of persuasion. Nor are they learned from repetition or gleaned from magical thinking. To deny objective truths is to be scientifically illiterate, not to be ideologically principled.

Differences in opinion enrich the diversity of a nation, and ought to be cherished and respected in any free society, provided everyone remains free to disagree with one another and, most importantly, everyone remains open to rational arguments that could change your mind.

Daily life in 1960 would be unrecognizable to anyone transported from the year 1930. From 1960 to 1990, a Cold War nuclear arms race between the United States and the Soviet Union threatens the survival of civilization. Though begun in the 1950s, the US stockpile of nuclear warheads peaks in the 1960s, with the Soviets’ stockpile peaking in the 1980s.11 The Berlin Wall, erected in 1962, becomes the greatest symbol of Winston Churchill’s “Iron Curtain,” separating Eastern from Western Europe. Yet it’s dismantled by 1989, as peace breaks out in Europe. The commercialization of the transistor allows consumer electronics to miniaturize, transforming audiovisual equipment from heavy, floor-mounted living room furniture to what you carry in your pocket.

When you chart the transmission curve for microwaves through Earth’s atmosphere,
you find a sudden, transparent window between the wavelengths of eighteen and
twenty-one centimeters. At either end of this little band, radio astronomers can
detect distinct emissions from the universe’s ubiquitous hydrogen atoms (H) and
their partners in water, hydroxyl molecules (OH). This band has been dubbed the
water hole—a phrase more commonly invoked for places on Earth where wild
creatures congregate to drink and wallow. We suspect that any aliens who know of
our existence and want to communicate with us might also know the absorptive
effects of water on various wavelengths. So if they’re clever, they might exploit the
dip and try to reach out to us via the microwave frequencies that get through the
water hole.
How about a less pacific result of the astrophysical discovery that water absorbs
microwaves? How difficult would it be to design a nonlethal weapon that targets the
water content of the human body? Three-fifths of our average body mass is water.
Such a weapon could operate on the same principle as the microwave oven.

When the CCD is exposed to light, whether from a nearby street scene or a faraway
galaxy, each of its pixels stores some number of electrons, depending on the
intensity of the light hitting each of the tiny locations on the CCD’s light-responsive
computer chip. The more intense the light, the more electrons get stored—although
if the light is too bright, you will saturate the detector and the excess electrons will
spill over into neighboring pixels, contaminating their data. Double the exposure,
and you get double the number of electrons. The electrons that congregate in each
pixel are then collected from the chip, tabulated, and turned into a single electronic
tile in the mosaic that constitutes the complete image. The more pixels, the more
resolution available to you. Nowadays you can easily download a street scene from
Wikimedia Commons that measures 2592 columns × 1944 rows, which translates
into a grid of more than 5,000,000 pixels—a crisply detailed photo. But that’s
nothing: if you’re not worried about overtaxing your computer, you can download an
image of the Orion Nebula from the HubbleSite Gallery that’s 18,000 × 18,000—a
grid of 324,000,000 pixels, packed to the gills with detail.
"""

print(clean_corpus(text_sample))

### 2.4 Segment and group

Same rules as the app's segmenter for prose: paragraphs split on blank lines, sentences
split after `.`, `!` or `?`, and sentences packed into segments of at most
`max_tokens_per_segment` tokens (counted with tiktoken `cl100k_base`). A sentence that alone
exceeds the budget is cut at `;`, `:` or `—` first, then at commas, then between words.
Segments are then grouped `segments_per_batch` at a time; every group is rendered in
section 3, so the output covers the whole text.

In [ ]:
token_encoder = tiktoken.get_encoding("cl100k_base")
sentence_boundary = re.compile(r"(?<=[.!?])\s+(?=[A-Z])|(?<=[.!?])\s*\n+", re.MULTILINE)


def count_tokens(text: str) -> int:
    return len(token_encoder.encode(text))


def pack(parts: list[str], budget: int) -> list[str]:
    chunks, current, current_tokens = [], [], 0
    for part in parts:
        tokens = count_tokens(part)
        if current and current_tokens + tokens > budget:
            chunks.append(" ".join(current))
            current, current_tokens = [], 0
        current.append(part)
        current_tokens += tokens
    return chunks + [" ".join(current)] if current else chunks


def pack_at_pauses(parts: list[str], budget: int) -> list[str]:
    chunks, current = [], []
    for part in parts:
        if count_tokens(part) <= budget:
            current.append(part)
            continue
        # A part with no pause left inside is broken between words.
        chunks += pack(current, budget) + pack(part.split(), budget)
        current = []
    return chunks + pack(current, budget)


def split_long_sentence(sentence: str, budget: int) -> list[str]:
    if count_tokens(sentence) <= budget:
        return [sentence]
    for pause in (r"(?<=[;:—])\s+", r"(?<=[,])\s+"):
        parts = re.split(pause, sentence)
        chunks = pack_at_pauses(parts, budget)
        if len(parts) > 1 and all(count_tokens(chunk) <= budget for chunk in chunks):
            return chunks
    return pack(sentence.split(), budget)


def segment_text(text: str, budget: int) -> list[str]:
    segments = []
    for paragraph in re.split(r"\n\s*\n", text):
        sentences = [s.strip() for s in sentence_boundary.split(paragraph.strip()) if s.strip()]
        segments += pack([piece for s in sentences for piece in split_long_sentence(s, budget)], budget)
    return segments


segments = segment_text(clean_corpus(text_sample), data_setup.max_tokens_per_segment)
groups = [segments[i:i + data_setup.segments_per_batch] for i in range(0, len(segments), data_setup.segments_per_batch)]

for index, segment in enumerate(segments[:10]):
    print(f"{index:3d}  {count_tokens(segment):3d} tokens  {segment[:90]}")
print(f"\n{len(segments)} segments in {len(groups)} groups of sizes {[len(g) for g in groups]}")

### 2.5 Cross lingual sentences

One sentence per language for section 3.9. With an English `reference_audio` this measures
how much of the reference accent leaks into other languages.

In [ ]:
cross_lingual_text = {
    "en": "The keeper made tea, because that was what one did on a night like this.",
    "fr": "Le gardien prépara du thé, car c'est ce que l'on fait par une nuit pareille.",
    "de": "Der Wärter kochte Tee, denn das tat man in einer solchen Nacht.",
    "es": "El farero preparó té, porque eso es lo que se hace en una noche así.",
    "it": "Il guardiano preparò il tè, perché è ciò che si fa in una notte simile.",
    "ja": "灯台守はお茶をいれた。こんな夜にはそうするものだからだ。",
    "zh": "守塔人泡了茶，因为这样的夜晚本该如此。"
}

## 3. Inference

| Variant | What it does | Why it is here |
|---|---|---|
| stock | joined prompt, unmodified upstream `generate` | the control, exactly today's behavior |
| joined | joined prompt, our sampler with the token cap exposed | shows whether the cut-off is only the cap |
| tensor | segments padded into one batch, one waveform each | every segment gets the full budget, one transformer pass per decode step |

Every variant renders every group of the text. Each snippet is written to `out_dir` with a
metrics row; section 3.7 stitches each variant back into one whole-text wav. On CPU the
whole text takes a long time per variant, so run 3.4 to 3.6 selectively if needed.

### 3.1 The renderers

Chatterbox produces audio in two stages: T3, a transformer, turns text tokens into speech
tokens at 25 per second, and S3Gen, the vocoder, turns speech tokens into a 24 kHz waveform.
Upstream `generate` runs T3 for one utterance with a hardcoded cap of 1000 speech tokens.
The joined and tensor variants reimplement the T3 sampling loop on the same weights so the cap
is exposed and several rows can be sampled at once.

**Result type and sampling knobs**

In [ ]:
speech_tokens_per_sec = 25
stock_token_cap = 1000


@dataclass
class Rendered:
    wav: torch.Tensor
    sample_rate: int
    speech_tokens: int
    truncated: bool
    inference_sec: float
    text: str

    @property
    def duration_sec(self) -> float:
        return self.wav.size(-1) / self.sample_rate


@dataclass
class SamplingParams:
    exaggeration: float = 0.5
    cfg_weight: float = 0.5
    temperature: float = 0.8
    repetition_penalty: float = 1.2
    min_p: float = 0.05
    top_p: float = 1.0
    max_new_tokens: int = 1000
    language_id: str | None = None


def is_multilingual(model: object) -> bool:
    return type(model).__name__ == "ChatterboxMultilingualTTS"

**Stock: upstream `generate`, unchanged.** It does not report its token count, so
`truncated` is inferred from the duration reaching the 1000 token cap.

In [ ]:
def render_stock(model: object, sentences: list[str], params: SamplingParams) -> Rendered:
    text = " ".join(s.strip() for s in sentences if s.strip())
    kwargs = {
        "exaggeration": params.exaggeration,
        "cfg_weight": params.cfg_weight,
        "temperature": params.temperature,
        "repetition_penalty": params.repetition_penalty,
        "min_p": params.min_p,
        "top_p": params.top_p
    }
    if is_multilingual(model):
        kwargs["language_id"] = params.language_id
    started = time.time()
    wav = model.generate(text, **kwargs)
    elapsed = time.time() - started
    tokens = round(wav.size(-1) / model.sr * speech_tokens_per_sec)
    truncated = tokens >= stock_token_cap - speech_tokens_per_sec
    return Rendered(wav, model.sr, tokens, truncated, elapsed, text)

**Text tokens and speaker conditioning.** Each text is tokenized and wrapped in start and
stop tokens. Rows of different length are right padded and carry a mask, so padding never
reaches the model. Exaggeration lives inside the conditioning, which is rebuilt only when
the requested value changes.

In [ ]:
def tokenize(model: object, text: str, language_id: str | None) -> torch.Tensor:
    if language_id:
        from chatterbox.mtl_tts import punc_norm
        tokens = model.tokenizer.text_to_tokens(punc_norm(text), language_id=language_id.lower())
    else:
        from chatterbox.tts import punc_norm
        tokens = model.tokenizer.text_to_tokens(punc_norm(text))
    tokens = F.pad(tokens, (1, 0), value=model.t3.hp.start_text_token)
    tokens = F.pad(tokens, (0, 1), value=model.t3.hp.stop_text_token)
    return tokens.to(model.device)[0]


def pad_tokens(rows: list[torch.Tensor], pad_value: int) -> tuple[torch.Tensor, torch.Tensor]:
    width = max(row.size(0) for row in rows)
    padded = torch.full((len(rows), width), pad_value, dtype=torch.long, device=rows[0].device)
    mask = torch.zeros((len(rows), width), dtype=torch.long, device=rows[0].device)
    for index, row in enumerate(rows):
        padded[index, :row.size(0)] = row
        mask[index, :row.size(0)] = 1
    return padded, mask


def apply_exaggeration(model: object, exaggeration: float) -> None:
    from chatterbox.models.t3.modules.cond_enc import T3Cond
    cond = model.conds.t3
    if float(exaggeration) == float(cond.emotion_adv[0, 0, 0].item()):
        return
    model.conds.t3 = T3Cond(
        speaker_emb=cond.speaker_emb,
        cond_prompt_speech_tokens=cond.cond_prompt_speech_tokens,
        emotion_adv=exaggeration * torch.ones(1, 1, 1)
    ).to(device=model.device)

**The prompt.** Conditioning, text and the start-of-speech token are embedded into one
prompt. With `cfg_weight > 0` every row is doubled: the second copy has its text zeroed, and
classifier-free guidance later pushes the output away from it. Position ids come from the
mask, so a padded row sees exactly the positions it would see alone.

In [ ]:
@dataclass
class PrefixState:
    embeds: torch.Tensor
    mask: torch.Tensor
    position_ids: torch.Tensor
    next_position: torch.Tensor


def prefix_embeds(t3: object, t3_cond: object, text_tokens: torch.Tensor, cfg_weight: float) -> torch.Tensor:
    batch = text_tokens.size(0)
    if cfg_weight > 0.0:
        text_tokens = torch.cat([text_tokens, text_tokens], dim=0)
    text_emb = t3.text_emb(text_tokens)
    if cfg_weight > 0.0:
        # Upstream zeroes the unconditional text before the position embedding.
        text_emb[batch:].zero_()
    text_emb = text_emb + t3.text_pos_emb(text_tokens)
    cond_emb = t3.prepare_conditioning(t3_cond)
    if cond_emb.size(0) != text_emb.size(0):
        cond_emb = cond_emb.expand(text_emb.size(0), -1, -1)
    return torch.cat([cond_emb, text_emb], dim=1)


def build_prefix(t3: object, t3_cond: object, text_tokens: torch.Tensor,
                 text_mask: torch.Tensor, params: SamplingParams) -> PrefixState:
    embeds = prefix_embeds(t3, t3_cond, text_tokens, params.cfg_weight)
    bos = torch.full((embeds.size(0), 1), t3.hp.start_speech_token, dtype=torch.long, device=t3.device)
    embeds = torch.cat([embeds, t3.speech_emb(bos) + t3.speech_pos_emb.get_fixed_embedding(0)], dim=1)
    if params.cfg_weight > 0.0:
        text_mask = torch.cat([text_mask, text_mask], dim=0)
    cond_len = embeds.size(1) - text_tokens.size(1) - 1
    ones = torch.ones((text_mask.size(0), cond_len), dtype=torch.long, device=text_mask.device)
    mask = torch.cat([ones, text_mask, torch.ones_like(text_mask[:, :1])], dim=1)
    return PrefixState(embeds, mask, (mask.cumsum(dim=1) - 1).clamp(min=0), mask.sum(dim=1, keepdim=True))

**The decode loop.** One transformer step per speech token for the whole batch. Each step
folds in guidance, applies repetition penalty, temperature, min-p and top-p, and samples.
Rows stop independently at their end-of-speech token; the loop ends when all have, or at
`max_new_tokens`.

In [ ]:
def step_logits(t3: object, embeds: torch.Tensor, mask: torch.Tensor,
                position_ids: torch.Tensor, past: object) -> tuple[torch.Tensor, object]:
    out = t3.tfmr(
        inputs_embeds=embeds, attention_mask=mask, position_ids=position_ids,
        past_key_values=past, use_cache=True, output_hidden_states=True, return_dict=True
    )
    return t3.speech_head(out.hidden_states[-1][:, -1, :]), out.past_key_values


def warp(logits: torch.Tensor, generated: torch.Tensor, batch: int, params: SamplingParams) -> torch.Tensor:
    from transformers.generation.logits_process import (
        MinPLogitsWarper, RepetitionPenaltyLogitsProcessor, TopPLogitsWarper
    )
    if params.cfg_weight > 0.0:
        cond, uncond = logits[:batch], logits[batch:]
        logits = cond + params.cfg_weight * (cond - uncond)
    logits = RepetitionPenaltyLogitsProcessor(penalty=float(params.repetition_penalty))(generated, logits)
    if params.temperature != 1.0:
        logits = logits / params.temperature
    logits = MinPLogitsWarper(min_p=params.min_p)(generated, logits)
    return TopPLogitsWarper(top_p=params.top_p)(generated, logits)


@torch.inference_mode()
def sample_batch(t3: object, t3_cond: object, text_tokens: torch.Tensor,
                 text_mask: torch.Tensor, params: SamplingParams) -> list[torch.Tensor]:
    batch, stop = text_tokens.size(0), t3.hp.stop_speech_token
    prefix = build_prefix(t3, t3_cond, text_tokens, text_mask, params)
    mask = prefix.mask
    logits, past = step_logits(t3, prefix.embeds, mask, prefix.position_ids, None)
    collected = [[] for _ in range(batch)]
    done = torch.zeros(batch, dtype=torch.bool, device=t3.device)
    generated = torch.full((batch, 1), t3.hp.start_speech_token, dtype=torch.long, device=t3.device)
    for step in range(params.max_new_tokens):
        probs = torch.softmax(warp(logits, generated, batch, params), dim=-1)
        tokens = torch.multinomial(probs, num_samples=1)
        tokens[done] = stop
        for row in range(batch):
            if not done[row]:
                collected[row].append(tokens[row, 0])
        done |= tokens.view(-1) == stop
        if bool(done.all()):
            break
        generated = torch.cat([generated, tokens], dim=1)
        embeds = t3.speech_emb(tokens) + t3.speech_pos_emb.get_fixed_embedding(step + 1)
        if params.cfg_weight > 0.0:
            embeds = torch.cat([embeds, embeds], dim=0)
        mask = torch.cat([mask, torch.ones_like(mask[:, :1])], dim=1)
        logits, past = step_logits(t3, embeds, mask, prefix.next_position + step, past)
    rows = [torch.stack(row) for row in collected]
    return [row[:-1] if row.numel() and int(row[-1]) == stop else row for row in rows]

**The vocoder.** Speech tokens go through S3Gen with the speaker's reference, then the same
watermark upstream applies. The multilingual model's last token before end-of-speech decodes
to about 40 ms of noise, so it is trimmed, as upstream does.

In [ ]:
@torch.inference_mode()
def decode_tokens(model: object, tokens: torch.Tensor) -> torch.Tensor:
    from chatterbox.models.s3tokenizer import drop_invalid_tokens
    tokens = drop_invalid_tokens(tokens)
    tokens = tokens[tokens < model.t3.hp.start_speech_token].to(model.device)
    wav, _ = model.s3gen.inference(speech_tokens=tokens, ref_dict=model.conds.gen)
    wav = wav.squeeze(0).detach().cpu().numpy()
    if is_multilingual(model):
        wav = wav[:max(1, int(tokens.shape[-1]) - 1) * (model.sr // speech_tokens_per_sec)]
    wav = model.watermarker.apply_watermark(wav, sample_rate=model.sr)
    return torch.from_numpy(wav).unsqueeze(0)

**Joined and tensor.** Joined samples the whole group as one row with the cap exposed. Tensor
samples every segment as its own row of one batch; the shared sampling time is split evenly
across rows.

In [ ]:
def render_joined(model: object, sentences: list[str], params: SamplingParams) -> Rendered:
    text = " ".join(s.strip() for s in sentences if s.strip())
    apply_exaggeration(model, params.exaggeration)
    padded, mask = pad_tokens([tokenize(model, text, params.language_id)], model.t3.hp.stop_text_token)
    started = time.time()
    speech = sample_batch(model.t3, model.conds.t3, padded, mask, params)[0]
    wav = decode_tokens(model, speech)
    tokens = int(speech.numel())
    return Rendered(wav, model.sr, tokens, tokens >= params.max_new_tokens, time.time() - started, text)


def render_tensor_batch(model: object, sentences: list[str], params: SamplingParams) -> list[Rendered]:
    texts = [s.strip() for s in sentences if s.strip()]
    if not texts:
        return []
    apply_exaggeration(model, params.exaggeration)
    rows = [tokenize(model, text, params.language_id) for text in texts]
    padded, mask = pad_tokens(rows, model.t3.hp.stop_text_token)
    started = time.time()
    speech = sample_batch(model.t3, model.conds.t3, padded, mask, params)
    share = (time.time() - started) / len(texts)
    return [
        Rendered(decode_tokens(model, row), model.sr, int(row.numel()),
                 int(row.numel()) >= params.max_new_tokens, share, text)
        for row, text in zip(speech, texts)
    ]

### 3.2 Settings: sampling

| Field | Meaning |
|---|---|
| `exaggeration` | emotion intensity, 0.5 is neutral |
| `cfg_weight` | how strongly the voice and pace follow the reference; 0 for cross lingual work |
| `temperature` | sampling randomness |
| `out_dir` | where snippets, whole-text wavs and metadata are written; a relative path is relative to the notebook's folder |
| `max_new_tokens` | speech token cap for joined and tensor, 25 tokens per second; upstream's 1000 is 40 seconds |

**QC settings.** A render fails QC when it is silent or holds one steady level with no speech in
it, then it is rendered again.

| Field | Meaning |
|---|---|
| `silence_dbfs` | fails as silent when even its loudest frames (95th percentile) stay below this level |
| `min_dynamic_range_db` | fails as steady noise when loud and quiet frames (95th vs 10th percentile) differ by less than this; speech rises and falls by 20 dB or more between syllables and pauses |
| `frame_ms` | frame length the levels are measured on |
| `max_retries` | extra attempts for a failed render; still failing after that, it is kept and flagged |

In [ ]:
params = SamplingParams(
    exaggeration=0.5,
    cfg_weight=0.5,
    temperature=0.8,
    max_new_tokens=1000,
    language_id=data_setup.language_id if model_setup.model_type != "english" else None
)
out_dir = Path("chatterbox_v3_out").absolute()
out_dir.mkdir(parents=True, exist_ok=True)


@dataclass
class QCSettings:
    silence_dbfs: float = -50.0
    min_dynamic_range_db: float = 12.0
    frame_ms: int = 50
    max_retries: int = 2


qc_settings = QCSettings()
params, qc_settings

### 3.3 Rendering helpers

**Audio files.** The model returns a float waveform. Like the app, it is turned into 16 bit
PCM (peak normalised only if it clips) and written as a WAV whose header carries the sample
rate, so every snippet and every stitched file plays as is with `Audio(path)` or any player.

In [ ]:
def to_pcm16(wav: torch.Tensor) -> np.ndarray:
    audio = wav.detach().float().cpu().squeeze()
    peak = float(audio.abs().max()) if audio.numel() else 0.0
    if peak > 1.0:
        audio = audio / peak
    return (audio * 32767).to(torch.int16).numpy()


def save_audio(file_path: Path, pcm: np.ndarray, sample_rate: int) -> None:
    sf.write(str(file_path), pcm, sample_rate, subtype="PCM_16")

**QC.** `check_audio` measures the level of every frame in dBFS and returns why the audio
failed, or an empty string when it looks like speech. `render_with_qc` renders a group and
re-renders whatever fails, up to `max_retries` times: the whole group for stock and joined,
only the failed segments for tensor.

In [ ]:
def frame_levels_db(pcm: np.ndarray, sample_rate: int, frame_ms: int) -> np.ndarray:
    frame = max(1, sample_rate * frame_ms // 1000)
    usable = len(pcm) // frame * frame
    if usable == 0:
        return np.array([-120.0])
    frames = pcm[:usable].astype(np.float64).reshape(-1, frame) / 32768
    return 20 * np.log10(np.maximum(np.sqrt((frames ** 2).mean(axis=1)), 1e-6))


def check_audio(pcm: np.ndarray, sample_rate: int, settings: QCSettings) -> str:
    levels = frame_levels_db(pcm, sample_rate, settings.frame_ms)
    loud = float(np.percentile(levels, 95))
    if loud < settings.silence_dbfs:
        return f"silent, loudest frames at {loud:.0f} dBFS"
    spread = loud - float(np.percentile(levels, 10))
    if spread < settings.min_dynamic_range_db:
        return f"steady level without speech, {spread:.0f} dB dynamic range"
    return ""


def as_parts(rendered: object) -> list:
    return rendered if isinstance(rendered, list) else [rendered]


def render_with_qc(variant: str, render: object, group: list[str]) -> tuple[list, list[int], list[str]]:
    parts = as_parts(render(model, group, params))
    attempts = [1] * len(parts)
    failures = [check_audio(to_pcm16(p.wav), p.sample_rate, qc_settings) for p in parts]
    for _ in range(qc_settings.max_retries):
        failed = [index for index, reason in enumerate(failures) if reason]
        if not failed:
            break
        print(f"{variant}: {len(failed)} part(s) failed QC ({failures[failed[0]]}), rendering again")
        retry_input = group if len(parts) == 1 else [parts[index].text for index in failed]
        for index, part in zip(failed, as_parts(render(model, retry_input, params))):
            parts[index], attempts[index] = part, attempts[index] + 1
            failures[index] = check_audio(to_pcm16(part.wav), part.sample_rate, qc_settings)
    return parts, attempts, failures

**Rows.** `record` saves one snippet and returns its metrics row: which group it came from, its
part within the group (stock and joined give one part per group, tensor gives one per segment),
the text it speaks, how many attempts it took and its QC verdict. For the tensor variant,
`inference_sec` is the shared sampling time divided across rows, which is the fair per segment
number. `render_variant` runs one variant over every group, in text order.

In [ ]:
def record(variant: str, group_index: int, part: int, rendered: object,
           peak_gb: float, attempts: int, failure: str) -> dict:
    name = f"{variant}_{group_index:03d}_{part:02d}.wav"
    save_audio(out_dir / name, to_pcm16(rendered.wav), rendered.sample_rate)
    return {
        "variant": variant,
        "group": group_index,
        "part": part,
        "file": name,
        "chars": len(rendered.text),
        "speech_tokens": rendered.speech_tokens,
        "duration_sec": round(rendered.duration_sec, 2),
        "inference_sec": round(rendered.inference_sec, 2),
        "rtf": round(rendered.inference_sec / max(rendered.duration_sec, 1e-6), 3),
        "truncated": rendered.truncated,
        "peak_mem_gb": round(peak_gb, 2),
        "over_weights_gb": round(max(peak_gb - weights_gb, 0.0), 2),
        "attempts": attempts,
        "qc": failure or "ok",
        "text": rendered.text
    }


def render_variant(variant: str, render: object) -> list[dict]:
    variant_rows = []
    for group_index, group in enumerate(groups):
        reset_peak_memory(device)
        parts, attempts, failures = render_with_qc(variant, render, group)
        peak_gb = peak_memory_gb(device)
        variant_rows += [
            record(variant, group_index, part, rendered, peak_gb, attempts[part], failures[part])
            for part, rendered in enumerate(parts)
        ]
        failed = sum(1 for reason in failures if reason)
        print(f"{variant}: group {group_index + 1}/{len(groups)} done" + (f", {failed} still failing QC" if failed else ""))
    return variant_rows


rows = []

### 3.4 Stock

In [ ]:
rows = [row for row in rows if row["variant"] != "stock"]
rows += render_variant("stock", render_stock)

### 3.5 Joined

In [ ]:
rows = [row for row in rows if row["variant"] != "joined"]
rows += render_variant("joined", render_joined)

### 3.6 Tensor batch

In [ ]:
rows = [row for row in rows if row["variant"] != "tensor"]
rows += render_variant("tensor", render_tensor_batch)

### 3.7 Whole text audio

Each variant's snippets are concatenated in text order into `<variant>_full.wav`, a 16 bit
PCM WAV like the snippets, the way
the app stitches a chapter (no gap by default, as in `stitch_chapter`). Every row gets a
`start_sec`: where its snippet begins inside the whole-text file, used by the transcript and
the phrase search in section 5.

In [ ]:
def stitch_variant(variant: str, gap_ms: int) -> Path:
    gap = np.zeros(model.sr * gap_ms // 1000, dtype=np.int16)
    parts = []
    offset_sec = 0.0
    for row in [row for row in rows if row["variant"] == variant]:
        pcm, sample_rate = sf.read(str(out_dir / row["file"]), dtype="int16")
        row["start_sec"] = round(offset_sec, 2)
        parts.extend([pcm, gap])
        offset_sec += (len(pcm) + len(gap)) / sample_rate
    full_path = out_dir / f"{variant}_full.wav"
    save_audio(full_path, np.concatenate(parts[:-1]), model.sr)
    return full_path


rendered_variants = list(dict.fromkeys(row["variant"] for row in rows))
full_audio = {variant: stitch_variant(variant, gap_ms=0) for variant in rendered_variants}
for variant, full_path in full_audio.items():
    info = sf.info(str(full_path))
    print(f"{variant:7s} {full_path.name}  {info.duration:.1f}s  {info.samplerate} Hz  {info.subtype}")

### 3.8 Token cap sweep

The model's own limit (`T3Config.max_speech_tokens`) is 4096, so the 1000 token cap is a choice, not a model limit.
Sweeping it on the joined variant over the first group shows the cliff: below the cap the
duration keeps growing, at the cap `truncated` flips and the tail is lost. Four renders, slow
on CPU.

In [ ]:
cap_rows = []
for cap in (500, 1000, 2000, 4096):
    rendered = render_joined(model, groups[0], replace(params, max_new_tokens=cap))
    pcm = to_pcm16(rendered.wav)
    save_audio(out_dir / f"cap_{cap}.wav", pcm, rendered.sample_rate)
    cap_rows.append({
        "cap": cap,
        "file": f"cap_{cap}.wav",
        "speech_tokens": rendered.speech_tokens,
        "duration_sec": round(rendered.duration_sec, 2),
        "truncated": rendered.truncated,
        "qc": check_audio(pcm, rendered.sample_rate, qc_settings) or "ok"
    })
print(f"{len(cap_rows)} caps rendered")

### 3.9 Cross lingual cloning

Multilingual only. The model card warns about accent bleed from the reference and
recommends `cfg_weight=0` for cross lingual work, which is what this cell uses.

In [ ]:
cross_rows = []
cross_languages = cross_lingual_text if model_setup.model_type != "english" else {}
if not cross_languages:
    print("english checkpoint has no language_id, set model_type='multilingual' in 1.2")
for lang, text in cross_languages.items():
    cross_params = replace(params, cfg_weight=0.0, max_new_tokens=2000, language_id=lang)
    for attempt in range(1, qc_settings.max_retries + 2):
        rendered = render_joined(model, [text], cross_params)
        pcm = to_pcm16(rendered.wav)
        failure = check_audio(pcm, rendered.sample_rate, qc_settings)
        if not failure:
            break
    save_audio(out_dir / f"cross_{lang}.wav", pcm, rendered.sample_rate)
    cross_rows.append({
        "lang": lang,
        "file": f"cross_{lang}.wav",
        "duration_sec": round(rendered.duration_sec, 2),
        "rtf": round(rendered.inference_sec / max(rendered.duration_sec, 1e-6), 3),
        "attempts": attempt,
        "qc": failure or "ok",
        "text": text
    })
print(f"{len(cross_rows)} languages rendered")

## 4. Result metrics

`rtf` is inference time over audio time: below 1 is faster than real time.
`over_weights_gb` is what rendering costs on top of the loaded weights.

In [ ]:
def print_table(table_rows: list[dict], columns: list[str]) -> None:
    widths = {col: max(len(col), *(len(str(row[col])) for row in table_rows)) for col in columns}
    print("  ".join(col.rjust(widths[col]) for col in columns))
    for row in table_rows:
        print("  ".join(str(row[col]).rjust(widths[col]) for col in columns))

### 4.1 Per segment

In [ ]:
print_table(rows, [
    "variant", "group", "part", "start_sec", "chars", "speech_tokens", "duration_sec",
    "inference_sec", "rtf", "truncated", "peak_mem_gb", "over_weights_gb", "attempts", "qc"
])

### 4.2 Per variant

The number that matters for the app is wall clock for the whole text and whether anything
was cut. A tensor batch that is slower per step can still win, because it pays the
transformer once for the whole batch instead of once per segment.

In [ ]:
def summarize(variant: str) -> dict:
    part = [row for row in rows if row["variant"] == variant]
    audio_sec = sum(row["duration_sec"] for row in part)
    wall_sec = sum(row["inference_sec"] for row in part)
    peak_gb = max(row["peak_mem_gb"] for row in part)
    return {
        "variant": variant,
        "audio_sec": round(audio_sec, 2),
        "wall_sec": round(wall_sec, 2),
        "rtf": round(wall_sec / max(audio_sec, 1e-6), 3),
        "truncated": f"{sum(row['truncated'] for row in part)}/{len(part)}",
        "retried": f"{sum(row['attempts'] > 1 for row in part)}/{len(part)}",
        "qc_failed": f"{sum(row['qc'] != 'ok' for row in part)}/{len(part)}",
        "peak_mem_gb": peak_gb,
        "over_weights_gb": round(max(peak_gb - weights_gb, 0.0), 2)
    }


variant_rows = [summarize(variant) for variant in rendered_variants]
print_table(variant_rows, list(variant_rows[0]))

### 4.3 Token cap sweep and cross lingual

In [ ]:
print_table(cap_rows, ["cap", "speech_tokens", "duration_sec", "truncated", "qc"])
if cross_rows:
    print()
    print_table(cross_rows, ["lang", "duration_sec", "rtf", "attempts", "qc"])

### 4.4 Save

Everything above, plus the settings that produced it, goes to
`out_dir/generation_metadata.json` so runs on different devices or versions can be compared
later.

In [ ]:
metadata = {
    "model": {**vars(model_setup), "device": device, "sample_rate": model.sr, "weights_gb": round(weights_gb, 2)},
    "data": vars(data_setup),
    "sampling": vars(params),
    "segments": rows,
    "variants": variant_rows,
    "token_caps": cap_rows,
    "cross_lingual": cross_rows
}
(out_dir / "generation_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
print(f"wrote {out_dir / 'generation_metadata.json'}")

## 5. Listen and analysis

### 5.1 Whole text, one player per variant

Each player is the full text for that variant. The transcript under it gives the start time
of every snippet, so you can seek straight to a passage.

In [ ]:
def print_transcript(variant: str) -> None:
    for row in [row for row in rows if row["variant"] == variant]:
        minutes, seconds = divmod(row["start_sec"], 60)
        flag = "" if row["qc"] == "ok" else f"  [QC FAILED: {row['qc']}]"
        print(f"  {int(minutes):02d}:{seconds:05.2f}  {row['text'][:110]}{flag}")


for variant, full_path in full_audio.items():
    print(f"{variant}, {sum(r['duration_sec'] for r in rows if r['variant'] == variant):.0f}s")
    display(Audio(str(full_path)))
    print_transcript(variant)

### 5.2 Find a passage

`listen_to` searches the snippet texts, case insensitive, and plays the matching snippet from
every variant next to each other, with its start time in the whole-text file. Change the
phrase and rerun.

In [ ]:
def listen_to(phrase: str) -> None:
    matches = [row for row in rows if phrase.lower() in row["text"].lower()]
    if not matches:
        print(f"no snippet contains '{phrase}'")
        return
    for row in matches:
        minutes, seconds = divmod(row["start_sec"], 60)
        hit = row["text"].lower().index(phrase.lower())
        context = row["text"][max(hit - 50, 0):hit + len(phrase) + 50]
        print(f"{row['variant']} snippet starts at {int(minutes):02d}:{seconds:05.2f} in {row['variant']}_full.wav, qc {row['qc']}: ...{context}...")
        display(Audio(str(out_dir / row["file"])))


listen_to("ice age")

### 5.3 Browse snippets of one variant

Every snippet of `browse_variant` in text order, each labelled with the text it speaks.

In [ ]:
browse_variant = "tensor"
for row in [row for row in rows if row["variant"] == browse_variant]:
    print(f"group {row['group']} part {row['part']}, {row['duration_sec']:.2f}s, qc {row['qc']}: {row['text'][:110]}")
    display(Audio(str(out_dir / row["file"])))

### 5.4 Token caps

In [ ]:
for row in cap_rows:
    print(f"cap {row['cap']}, truncated={row['truncated']}, qc {row['qc']}")
    display(Audio(str(out_dir / row["file"])))

### 5.5 Cross lingual

In [ ]:
for row in cross_rows:
    print(f"{row['lang']}, qc {row['qc']}: {row['text'][:60]}")
    display(Audio(str(out_dir / row["file"])))

### 5.6 What to judge

1. **Cut-offs.** Does stock lose the end of groups (check the transcript in 5.1), and does raising the cap on joined fix
   it, or does quality fall apart before the audio does?
2. **Prosody across boundaries.** The joined variants hear the whole group as one utterance,
   so sentence transitions should flow. The tensor variant renders segments independently,
   which can sound clipped or over-reset at each boundary.
3. **Speaker drift.** Compare the start and end of each whole-text file. Independent sampling from the
   same conditioning can drift in pitch or pace.
4. **Throughput.** Table 4.2. Tensor batching wins when the per step transformer cost
   dominates, which is the CUDA case; on CPU the gap is smaller.
5. **Memory.** `over_weights_gb` is what batching costs. A tensor batch that does not fit
   under `vram_cap_gb` is not a candidate, whatever its RTF.
6. **Cross lingual.** How much of the reference accent leaks, and whether `cfg_weight=0` is
   enough to keep it out.

### 5.7 Findings

Record the verdict here, with the device and `mtl_version` it was measured on, before the
pipeline is changed.

- Device / version:
- Cut-offs:
- Prosody:
- Drift:
- Throughput:
- Memory:
- Cross lingual:
- Decision: